---
title: |
  Transformers for Sequence Data
subtitle: |
  Specialized neural network modules for processing data sequences. 
jupyter: python3
---

{{< include ../assets/includes/_colab-link.qmd >}}

::: {.content-hidden}

## Boilerplate preamble for LaTeX macros and Python viz style
$$
{{< include ../assets/includes/_macros.tex >}}
$$

In [308]:
import style

:::


[Much of the code for this lecture is based on [this tutorial](https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch) by Arjun Sarkar for DataCamp.]{.aside} 

In [309]:
import torch 
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from torch import nn
import urllib
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
!pip install torchinfo
from torchinfo import summary


In [310]:
url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes-update/refs/heads/main/data/seuss.txt"
text = "\n".join([line.decode('utf-8').strip() for line in urllib.request.urlopen(url)])
# text = text[:10000]  # use only the first 10k characters for faster training

In [311]:
text = text[:1000]

## Data and Embedding

In [312]:
#| code-fold: true
# construct tokenizer
from tokenizers import Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

# dataset for embedding
from torch.utils.data import Dataset, DataLoader
class CBOWDataset(Dataset):
    def __init__(self, tokens, context_length):
        self.tokens = tokens
        
        # dict to remap tokens to contiguous integers
        self.token_to_idx = {token: i for i, token in enumerate(set(tokens))}
        self.idx_to_token = {i: token for token, i in self.token_to_idx.items()}
        
        # context length is the number of tokens on either side of the target token
        self.context_length = context_length
        self.data = []
        for i in range(context_length, len(tokens)-context_length):
            for j in range(-context_length, context_length+1):
                if j != 0:
                    self.data.append((self.token_to_idx[tokens[i+j]], self.token_to_idx[tokens[i]]))
                    self.data.append((self.token_to_idx[tokens[i+j]], self.token_to_idx[tokens[i]]))
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), torch.tensor(self.data[idx][1])

# CBOW embedding model
class CBOW(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_embedding)
        self.linear = nn.Linear(d_embedding, vocab_size)
        
    def forward(self, x):
        embedded = self.embeddings(x) 
        output = self.linear(embedded)
        return output

# for embedding visualization after training

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    checkpoint = "openai-community/gpt2"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)


    # construct dataset, dataloader, model
    context_length = 5
    tokens = tokenizer.encode(text)
    data = CBOWDataset(tokens, context_length=context_length)
    dataloader = DataLoader(data, batch_size=32, shuffle=True)
    vocab_size = len(data.token_to_idx)
    d_embedding = 8
    model = CBOW(vocab_size, d_model=d_embedding)

    # construct training loss and optimizer

    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    model.to(device)

    # training loop
    for epoch in range(5):
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            opt.step()

In [313]:
embedding_layer = model.embeddings
embedding_dim = embedding_layer.weight.shape[1]


In [314]:
class SeussDataSet(Dataset):
    def __init__(self, tokens, context_length = 5): 
        self.context_length = context_length
        self.tokens = tokens
        self.vocab_length = len(set(tokens))
        self.idx_to_token = {i: token for i, token in enumerate(set(tokens))}
        self.token_to_idx = {token: i for i, token in enumerate(set(tokens))}

    def __len__(self):
        return len(self.tokens) - self.context_length - 1
    
    def __getitem__(self, key):
        
        source = self.tokens[key:(self.context_length + key)]
        target = self.tokens[(key+1):(self.context_length + key+1)] # shift right?
        
        source = torch.tensor([self.token_to_idx[token] for token in source], dtype=torch.long)
        target = torch.tensor([self.token_to_idx[token] for token in target], dtype=torch.long)
        
        return source.to(device), target.to(device)
    
class SeussDecoder: 
    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.idx_to_token = dataset.idx_to_token
        self.token_to_idx = dataset.token_to_idx
        self.tokenizer = tokenizer
    
    def decode(self, token_indices):
        tokens = [self.idx_to_token[idx] for idx in token_indices]
        return self.tokenizer.decode(tokens)

In [315]:
context_length = 10
data = SeussDataSet(tokens, context_length = context_length)
decoder = SeussDecoder(data, tokenizer)
print(f"Number of training examples: {len(data)}")
print(f"Vocabulary size: {data.vocab_length}")

Number of training examples: 322
Vocabulary size: 121


In [316]:
# basic transformer model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor([10000.0]))) / d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers):
        super().__init__()
        
        self.embeddings = embedding_layer
        self.embeddings.requires_grad_(False)  # freeze the embedding layer
        
        self.pe = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_layer = nn.Linear(d_model, vocab_size)
    
    def forward(self, source, target):
        source_embedded = self.embeddings(source) 
        target_embedded = self.embeddings(target)
        source_pe = self.pe(source_embedded)
        target_pe = self.pe(target_embedded)
        encoded = self.transformer_encoder(source_pe)
        decoded = self.transformer_decoder(target_pe, encoded)
        output = self.output_layer(decoded)
        return output

In [317]:
# instantiate and train the transformer model

dataloader = DataLoader(data, batch_size=128, shuffle=True)
vocab_size = len(data.token_to_idx)
d_model = embedding_layer.weight.shape[1]
n_heads = 8
n_layers = 2
model = TransformerModel(vocab_size, d_model, n_heads, n_layers)
model = model.to(device)

/var/folders/xn/wvbwvw0d6dx46h9_2bkrknnw0000gn/T/ipykernel_5078/194520722.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [318]:
# x = data[0][0].unsqueeze(0).to(device)  # example input
# summary(model, input_size=(x.shape), device=device, dtypes=[torch.long])

In [327]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(100):
    total_loss = 0
    for source, target in dataloader:
        source, target = source.to(device), target.to(device)
        opt.zero_grad()
        output = model(source, target)
        pred   = output[:,0,:]
        actual = target[:,0]
        loss   = loss_fn(pred, actual)
        loss.backward()
        total_loss += loss.item()
        opt.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss / len(dataloader)}")

Epoch 0, Loss: 3.045811335245768
Epoch 10, Loss: 3.070502440134684
Epoch 20, Loss: 2.8352564175923667
Epoch 30, Loss: 2.95659867922465
Epoch 40, Loss: 2.7334792613983154
Epoch 50, Loss: 2.6506696542104087
Epoch 60, Loss: 2.6342338720957437
Epoch 70, Loss: 2.7218617598215737
Epoch 80, Loss: 2.4833339055379233
Epoch 90, Loss: 2.541383425394694


# Text generation from the model

In [328]:
def boltzmann_prediction(preds, temperature = 1.0):
    probabilities = torch.nn.Softmax(dim = 0)(preds / temperature)
    return torch.multinomial(probabilities, num_samples=1).item()

In [329]:
def generate_text(model, decoder, start_tokens, max_length=20, temperature=1.0):
    generated_tokens = start_tokens.copy()
    
    for _ in range(max_length):
        input_tensor = torch.tensor(generated_tokens[-(context_length+1):-1], dtype=torch.long).unsqueeze(0).to(device) 
        target_tensor = torch.tensor(generated_tokens[-context_length:], dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            preds = model(input_tensor, target_tensor).squeeze(0) 
        
        next_token = boltzmann_prediction(preds[0], temperature) 
        generated_tokens.append(next_token) 
    
    return decoder.decode(generated_tokens)

In [330]:
start_tokens = torch.tensor([data.token_to_idx[token] for token in tokens[:context_length+1]])
dr_sus = generate_text(model, decoder, start_tokens.tolist(), max_length=50, temperature=.2)
print(dr_sus)

The Cat in the Hat

By Dr. Se"Sit we
!




 to
 all
!
 us!!! to
 all


!!! we to
 all


! we we we to we I


!!!We


## Sequence Data

## The Attention Mechanism

The use of the attention mechanism for natural language processing took off with with a seminal paper by @vaswani2017attention, which integrated the attention mechanism into a neural network architecture called the *transformer*. 

The attention mechanism is a technical expression of a simple idea: 


::: {#fig-attention-diagrams}

![](fig/attention-heads.png)


Schematic diagrams of two attention heads performing different functions. The first attention on the left picks up referential relationships between words, while the one on the right has been trained to detect rhyming words. 

:::

### Dr. Seuss Example

In [323]:
import urllib
url = "https://raw.githubusercontent.com/middcs/data-science-notes/refs/heads/main/data/seuss/places.txt"
text = "\n".join([line.decode('utf-8').strip() for line in urllib.request.urlopen(url)])
print(text[0:193])

Congratulations!
Today is your day.
You're off to Great Places!
You're off and away!

You have brains in your head.
You have feet in your shoes.
You can steer yourself
any direction you choose.


Dataset code from previous lecture notes. 

In [324]:
class TextDataSet(Dataset):
    def __init__(self, text, tokenizer = Tokenizer(BPE()), trainer = BpeTrainer(min_frequency=10), context_length = 5): 
        self.context_length = context_length
        self.text = text
        self.tokenizer = tokenizer
        self.tokenizer.train_from_iterator([text], trainer)
        self.tokens = self.tokenizer.encode(text).ids
        self.vocab_length = self.tokenizer.get_vocab_size()

    def __len__(self):
        return len(self.tokens) - self.context_length#<1>
    
    def __getitem__(self, key):
        target = torch.tensor(self.tokens[self.context_length + key])#<2>
        features = self.tokens[key:(self.context_length + key)]#<3>

        feature_tensor = torch.tensor(features, dtype=torch.long)
        feature_text = self.tokenizer.decode(features)#<4>
        target_text = self.tokenizer.decode([target])

        return feature_tensor.to(device), target.to(device), feature_text, target_text


NameError: name 'BPE' is not defined

In [ ]:
data = TextDataSet(
    text, 
    tokenizer = tokenizer, 
    trainer = trainer, 
    context_length = 5
    )
print(f"Number of training examples: {len(data)}")
print(f"Vocabulary size: {data.vocab_length}")

### Key-Query-Value (KQV) Attention

[The mathematical exposition of the KQV attention mechanism is based on Chapter 12.1 of @bishop2023deep.]{.aside}

In [ ]:
class AttentionHead(torch.nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_k)

    def forward(self, x):
        Q = self.W_q(x)  # Queries
        K = self.W_k(x)  # Keys
        V = self.W_v(x)  # Values

        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (K.size(-1) ** 0.5)
        attention_weights = torch.nn.functional.softmax(scores, dim=-1)

        # Compute the output as a weighted sum of values
        output = torch.matmul(attention_weights, V)
        return output

In [ ]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return x

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, d_k, num_heads):
        super().__init__()
        self.heads = torch.nn.ModuleList([AttentionHead(d_model, d_k) for _ in range(num_heads)])
        self.linear = torch.nn.Linear(num_heads * d_k, d_model)

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]
        concatenated = torch.cat(head_outputs, dim=-1)
        output = self.linear(concatenated)
        return output

### Self-Attention

## A First Attention-Based Model

## Towards Transformers

## References


## New Outline 

- Math description of attention heads, maybe causal masking
- Gesture towards positional encoding
- Big-picture transformer architecture: encoder-decoder as generating a memory 
- Grab GPT-2 and pick it apart, similar to 1010 lecture by Michael